In [9]:
# Install Kaggle API and download dataset
!pip install kaggle -q

# Upload your kaggle.json API key when prompted
from google.colab import files
files.upload()  # upload kaggle.json from your PC

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d jayaprakashpondy/blood-cancer-dataset --unzip

cp: cannot stat 'kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/jayaprakashpondy/blood-cancer-dataset
License(s): CC0-1.0
 99% 1.66G/1.68G [00:06<00:00, 274MB/s]
100% 1.68G/1.68G [00:06<00:00, 288MB/s]


In [10]:
import os

# Paste your NEW key values here after revoking the exposed one
os.environ['KAGGLE_USERNAME'] = 'saiharshith002'
os.environ['KAGGLE_KEY']      = 'KGAT_58c252b3161a473b41a08f228cad1ac3'

!pip install kaggle -q
!kaggle datasets download -d jayaprakashpondy/blood-cancer-dataset --unzip -p /content/
!ls /content/

Dataset URL: https://www.kaggle.com/datasets/jayaprakashpondy/blood-cancer-dataset
License(s): CC0-1.0
 99% 1.67G/1.68G [00:09<00:00, 236MB/s]
100% 1.68G/1.68G [00:09<00:00, 187MB/s]
Dataset  sample_data


In [2]:
# For Data Processing
import numpy as np
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from PIL import Image, ImageEnhance

# For ML Models
from tensorflow import keras
from tensorflow.keras.layers import *
from tensorflow.keras.losses import *
from tensorflow.keras.models import *
from tensorflow.keras.metrics import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.applications import *
from tensorflow.keras.preprocessing.image import load_img

# For Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Miscellaneous
from tqdm import tqdm
import os
import random

# <b>2 <span style='color:#4285f4'>|</span> Reading the Dataset</b>

In [4]:
train_dir = '/content/Dataset'
test_dir = '/content/Dataset'

train_paths = []
train_labels = []

for label in os.listdir(train_dir):
    label_path = os.path.join(train_dir, label)
    if os.path.isdir(label_path): # Check if it's a directory
        for image in os.listdir(label_path):
            train_paths.append(os.path.join(label_path, image))
            train_labels.append(label)

train_paths, train_labels = shuffle(train_paths, train_labels)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Dataset'

In [ ]:
print("Contents of /content/Dataset:")
!ls -R /content/Dataset

# Please run this cell and provide the output so I can help you correct the paths for train_dir and test_dir.

In [ ]:
plt.figure(figsize=(10,10))
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99']
plt.pie([len(os.listdir(train_dir+"//" +label))
        for label in os.listdir(train_dir)],
         labels=os.listdir(train_dir),
          colors = colors,
           explode = (0.08,0.08,0.08,0.08)
           ,autopct='%1.1f%%')
plt.show()


plt.figure(figsize=(14,6))
sns.countplot(x=train_labels, palette=colors, hue=train_labels, legend=False)
plt.title('Distribution of the types of brain tumors')
plt.xlabel('Type of brain tumor')
plt.ylabel('Number of images');

In [ ]:
test_paths = []
test_labels = []

for label in os.listdir(test_dir):
    for image in os.listdir(test_dir+ '//' + label):
        test_paths.append(test_dir +'//'+label + '//'+image)
        test_labels.append(label)

test_paths, test_labels = shuffle(test_paths, test_labels)

In [ ]:
def preprocess_image(image):
    """Plain normalization — used for validation and test, no randomness."""
    return np.array(image) / 255.0

def augment_image(image):
    """Random augmentation — used for training only."""
    image = Image.fromarray(np.uint8(image))
    image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
    image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))
    image = ImageEnhance.Sharpness(image).enhance(random.uniform(0.8, 1.2))
    image = np.array(image) / 255.0
    return image

In [ ]:
IMAGE_SIZE = 128

def open_images(paths, augment=True):
    images = []
    for path in paths:
        image = load_img(path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
        if augment:
            image = augment_image(image)
        else:
            image = preprocess_image(image)
        images.append(image)
    return np.array(images)

# Preview a few training samples
images = open_images(train_paths[50:58])
labels = train_labels[50:58]
fig = plt.figure(figsize=(12, 6))
for x in range(1, 9):
    fig.add_subplot(2, 4, x)
    plt.axis('off')
    plt.title(labels[x-1])
    plt.imshow(images[x-1])
plt.rcParams.update({'font.size': 12})
plt.show()

In [ ]:
unique_labels = os.listdir(train_dir)

def encode_label(labels):
    encoded = []
    for x in labels:
        encoded.append(unique_labels.index(x))
    return np.array(encoded)

def decode_label(labels):
    decoded = []
    for x in labels:
        decoded.append(unique_labels[x])
    return np.array(decoded)

def datagen(paths, labels, batch_size=12, epochs=1, augment=True):
    for _ in range(epochs):
        for x in range(0, len(paths), batch_size):
            batch_paths = paths[x:x+batch_size]
            batch_images = open_images(batch_paths, augment=augment)
            batch_labels = encode_label(labels[x:x+batch_size])
            yield batch_images, batch_labels

EfficientNetB3 Model

### using transfer learning:

In [ ]:
from tensorflow.keras.applications import EfficientNetB3

base_model = EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    pooling='max'
)

# Freeze base for Phase 1 — only the custom head will be trained
base_model.trainable = False

model = Sequential([
    base_model,
    BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001),
    Dense(256, activation='relu'),
    Dropout(rate=0.45, seed=123),
    Dense(len(unique_labels), activation='softmax')
])

model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Validation split (15% of training data)
val_size = int(len(train_paths) * 0.15)
val_paths  = train_paths[:val_size]
val_labels = train_labels[:val_size]
fit_paths  = train_paths[val_size:]
fit_labels = train_labels[val_size:]

print(f"Train samples : {len(fit_paths)}")
print(f"Val   samples : {len(val_paths)}")

# Class weights to handle imbalance
class_indices = encode_label(fit_labels)
class_weights = compute_class_weight('balanced', classes=np.unique(class_indices), y=class_indices)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

In [ ]:
# Phase 1 compile — higher LR is fine when base is frozen
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['sparse_categorical_accuracy']
)

Train EfficientNetB3 Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1)
]

batch_size   = 20
train_steps  = int(len(fit_paths) / batch_size)
val_steps    = max(1, int(len(val_paths) / batch_size))

# ── Phase 1: Train head only (base frozen) ──────────────────────────────────
print("=== Phase 1: Training head (base frozen) ===")
history1 = model.fit(
    datagen(fit_paths, fit_labels, batch_size=batch_size, epochs=15, augment=True),
    epochs=15,
    steps_per_epoch=train_steps,
    validation_data=datagen(val_paths, val_labels, batch_size=batch_size, epochs=15, augment=False),
    validation_steps=val_steps,
    callbacks=callbacks
)

# ── Phase 2: Fine-tune entire model at very low LR ──────────────────────────
print("\n=== Phase 2: Fine-tuning entire model ===")
base_model.trainable = True
model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['sparse_categorical_accuracy']
)

history2 = model.fit(
    datagen(fit_paths, fit_labels, batch_size=batch_size, epochs=20, augment=True),
    epochs=20,
    steps_per_epoch=train_steps,
    validation_data=datagen(val_paths, val_labels, batch_size=batch_size, epochs=20, augment=False),
    validation_steps=val_steps,
    callbacks=callbacks
)

In [ ]:
def plot_history(history, title):
    acc_key  = 'sparse_categorical_accuracy'
    val_key  = 'val_sparse_categorical_accuracy'
    epochs   = range(1, len(history.history[acc_key]) + 1)
    plt.figure(figsize=(14, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history[acc_key],     'b-o', label='Train Acc')
    plt.plot(epochs, history.history[val_key],     'g-o', label='Val Acc')
    plt.title(f'{title} — Accuracy')
    plt.xlabel('Epoch'); plt.legend(); plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history['loss'],      'r-o', label='Train Loss')
    plt.plot(epochs, history.history['val_loss'],  'orange', label='Val Loss', marker='o')
    plt.title(f'{title} — Loss')
    plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
    plt.show()

plot_history(history1, 'Phase 1 (frozen base)')
plot_history(history2, 'Phase 2 (full fine-tune)')

# <b>7.2 <span style='color:#4285f4'>|</span> Evaluate Model with Test Samples</b>

In [ ]:
batch_size = 32
steps = int(len(test_paths) / batch_size)
y_pred = []
y_true = []

# augment=False — deterministic preprocessing during evaluation
for x, y in tqdm(datagen(test_paths, test_labels, batch_size=batch_size, epochs=1, augment=False), total=steps):
    pred = model.predict(x)
    pred = np.argmax(pred, axis=-1)
    for i in decode_label(pred):
        y_pred.append(i)
    for i in decode_label(y):
        y_true.append(i)

In [ ]:
print(classification_report(y_true, y_pred))

### Save the entire model (architecture + weights) to Google Drive

In [ ]:
import os

# Define the path to save the entire model
model_save_path = '/content/drive/MyDrive/my_blood_cancer_model.keras'

# Save the entire model in the SavedModel format
# This saves both the architecture and the weights
model.save(model_save_path)

print(f"Entire model saved to: {model_save_path}")

### How to Download to your Local Drive

1.  **Open Google Drive**: Go to [drive.google.com](https://drive.google.com/) in your web browser.
2.  **Navigate to the folder**: Look for the `my_blood_cancer_model` folder (if you saved the entire model) or the `model.weights.h5` file in your 'My Drive' folder.
3.  **Download**: Right-click on the file or folder and select 'Download'. If you downloaded the folder, it will likely be compressed into a `.zip` file. You will need to extract it on your local machine.

### How to Load the Saved Model in your Project

Once downloaded, you can load the model in your Python project using Keras:

If you saved **only the weights** (`model.weights.h5`):
```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow.keras.applications import EfficientNetB3

# You need to re-create the model architecture exactly as it was defined before saving weights
IMAGE_SIZE = 128 # Make sure this matches the size used during training
NUM_CLASSES = 4 # Make sure this matches the number of unique_labels

base_model_loaded = EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    pooling='max'
)
base_model_loaded.trainable = False # Or True, depending on how you want to use it

loaded_model = Sequential([
    base_model_loaded,
    BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001),
    Dense(256, activation='relu'),
    Dropout(rate=0.45, seed=123),
    Dense(NUM_CLASSES, activation='softmax')
])

# Load the weights into the re-created model
loaded_model.load_weights('path/to/your/local/model.weights.h5')

print("Model weights loaded successfully.")
```

If you saved the **entire model** (`my_blood_cancer_model` folder):
```python
import tensorflow as tf

# Load the entire model from the SavedModel format
loaded_model = tf.keras.models.load_model('path/to/your/local/my_blood_cancer_model')

print("Entire model loaded successfully.")
loaded_model.summary()
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_weights('/content/drive/MyDrive/model.weights.h5')
print("Model saved to Google Drive")

In [ ]:
print(unique_labels)

In [5]:
import json
print(unique_labels)

# Also save it so you never have to worry again
with open('labels.json', 'w') as f:
    json.dump(unique_labels, f)
print("Saved labels.json")

NameError: name 'unique_labels' is not defined

In [11]:
import os, json

# Make sure the dataset is still there
labels = os.listdir('/content/Dataset')
print(labels)

with open('labels.json', 'w') as f:
    json.dump(labels, f)

print("Saved labels.json")

['[Malignant] early Pre-B', '[Malignant] Pre-B', '[Malignant] Pro-B', 'Benign']
Saved labels.json


In [12]:
from google.colab import files
files.download('labels.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# Save in legacy H5 format (compatible with older TF/Keras)
model.save('blood_cancer_model.h5')
print("Saved")

from google.colab import files
files.download('blood_cancer_model.h5')


NameError: name 'model' is not defined

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf

# Load the saved model from Drive
model = tf.keras.models.load_model('/content/drive/MyDrive/my_blood_cancer_model.keras')
print("Model loaded")

# Re-save in legacy H5 format (compatible with your local TF)
model.save('blood_cancer_model.h5')
print("Saved as H5")

from google.colab import files
files.download('blood_cancer_model.h5')

Mounted at /content/drive


Model loaded
Saved as H5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>